In [31]:
!pip install geopandas
!pip install shapely
!pip install altair
!pip install pandas
!pip install requests
!pip install vl-convert-python
!pip install python-dotenv
!pip install geopy requests

In [32]:
import geopandas as gpd
import pandas as pd
import requests
import altair as alt
from shapely.geometry import shape
import datetime
import os
import json
from shapely.geometry import shape, Point
import vl_convert 
from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from shapely.geometry import shape
import isochrone_api as isapi

In [33]:
load_dotenv()  # loads variables from .env into environment

NAVITIA_TOKEN = os.getenv("NAVITIA_TOKEN")
print(f"Token loaded: {'✅' if NAVITIA_TOKEN else '❌ Not found'}")
print(f"Navitia Token: {NAVITIA_TOKEN[:4]}...{NAVITIA_TOKEN[-4:]}")  # Print only the first and last 4 characters for verification

Token loaded: ✅
Navitia Token: 0BwL...wLQ1


In [34]:
ISOCHRONE_PATH = "../isochrone/"
CAMPUS_ADDRESS = "Université Paris 1 Panthéon-Sorbonne, Paris, France"
CAMPUS_NAME = "Paris 1 Panthéon-Sorbonne"
TIME_STUDIED = 60  # minutes

# Skip geocoder — hardcode correct coordinates for Paris 1 (Place du Panthéon)
start_lon = 2.3288788389736297
start_lat = 48.85380910031158

print(f"Using hardcoded coordinates for Paris 1: {start_lon}, {start_lat}")

Using hardcoded coordinates for Paris 1: 2.3288788389736297, 48.85380910031158


In [35]:
#if cached file exists, load isochrone from there; otherwise, fetch from Navitia API and save to cache
cached_file = f"../isochrone/isochrone_paris_{TIME_STUDIED}min.json"
if os.path.exists(cached_file):
    print(f"Loading isochrone data from cache: {cached_file}")
    with open(cached_file, "r") as f:
        data = json.load(f)
else:
    print(f"Fetching isochrone data from Navitia API for {TIME_STUDIED}-min commute...")
    isapi.call_API(TIME_STUDIED, NAVITIA_TOKEN, start_lon, start_lat)
    with open(cached_file, "r") as f:
        data = json.load(f)

Fetching isochrone data from Navitia API for 60-min commute...
🌐 Making API call...
Status code: 200
✅ Response saved to ../isochrone/isochrone_paris_60min.json

Top-level keys: ['feed_publishers', 'links', 'isochrones', 'warnings', 'context']
Number of isochrone zones returned: 1
Keys in first isochrone object: ['geojson', 'max_duration', 'min_duration', 'from', 'requested_date_time', 'min_date_time', 'max_date_time']

First isochrone (without geometry):
{
  "max_duration": 3600,
  "min_duration": 0,
  "from": {
    "id": "2.328387;48.854141",
    "name": "27 Rue Saint-Guillaume (Paris)",
    "quality": 0,
    "embedded_type": "address",
    "address": {
      "id": "2.328387;48.854141",
      "name": "Rue Saint-Guillaume",
      "house_number": 27,
      "coord": {
        "lon": "2.328387",
        "lat": "48.854141"
      },
      "label": "27 Rue Saint-Guillaume (Paris)",
      "administrative_regions": [
        {
          "id": "admin:fr:75056",
          "name": "Paris",
     

In [36]:
#we get all files from the isochrone folder
isochrone_files = [f for f in os.listdir(ISOCHRONE_PATH) if f.endswith(".json")]
display(isochrone_files)

['isochrone_paris_45min.json',
 'isochrone_paris_60min.json',
 'isochrone_paris_30min.json',
 'isochrone_paris_15min.json']